In [9]:
import requests
from selenium import webdriver
from bs4 import BeautifulSoup
import re
import pandas as pd
import time

url = "https://www.nykaa.com/mom-baby/baby-care/c/14798"

headers = {
    "User-Agent": "Mozilla/5.0"
}

driver = webdriver.Chrome()
driver.get(url)

time.sleep(5)

print(driver.current_url)
print(driver.title)

https://www.nykaa.com/mom-baby/baby-care/c/14798
Buy Baby Products Online With Discounts Upto 70% And Above In India


In [10]:
response = requests.get(url, headers=headers)

print("Length:", len(response.content))

Length: 413


In [8]:
soup = BeautifulSoup(driver.page_source, "html.parser")

titles = soup.find_all("h2", class_="css-xrzmfa")

print("Number of products:", len(titles))

Number of products: 20


In [12]:
# Empty list to store product data
products = []

for product in titles[:20]:

    # Product Name
    product_name = product.get_text(strip=True)

    # Product card text
    parent = product.parent
    text = parent.get_text(" ", strip=True)

    # MRP
    mrp_match = re.search(r"Regular price ₹(\d+)", text)

    if mrp_match:
        mrp = mrp_match.group(1)
    else:
        mrp = "N/A"

    # Selling Price
    price_match = re.search(r"Discounted price ₹(\d+)", text)

    if price_match:
        price = price_match.group(1)
    else:
        price = "N/A"

    # Discount
    discount_match = re.search(r"(\d+)% Off", text)

    if discount_match:
        discount = discount_match.group(1) + "%"
    else:
        discount = "N/A"

    # Brand
    brand_match = re.match(r"(\S+)", product_name)

    if brand_match:
        brand = brand_match.group(1)
    else:
        brand = "N/A"

    # Rating
    rating = "N/A"

    current = product

    for i in range(5):

        if current:

            current_text = current.get_text(" ", strip=True)

            rating_match = re.search(r"\b[0-5]\.\d\b", current_text)

            if rating_match:
                rating = rating_match.group(0)
                break

            current = current.parent

    # Reviews
    reviews_match = re.search(r"\(\s*(\d+)\s*\)", text)

    if reviews_match:
        reviews = reviews_match.group(1)
    else:
        reviews = "N/A"

    # Product URL
    link = product.find_parent("a", href=True)

    if link:
        href = link["href"]

        if href.startswith("http"):
            product_url = href
        else:
            product_url = "https://www.nykaa.com" + href
    else:
        product_url = "N/A"

    # Category
    category = "Baby"

    # Store product data
    products.append({
        "Product Name": product_name,
        "Brand": brand,
        "MRP": mrp,
        "Selling Price": price,
        "Discount": discount,
        "Rating": rating,
        "Reviews": reviews,
        "Product URL": product_url,
        "Category": category
    })

print("Products collected:", len(products))

Products collected: 20


In [14]:
df = pd.DataFrame(products)

df.to_csv("nykaa_baby_products.csv", index=False)

print("Baby CSV created successfully!")
print("Total products saved:", len(df))

Baby CSV created successfully!
Total products saved: 20


In [15]:
df

,Product Name,Brand,MRP,Selling Price,Discount,Rating,Reviews,Product URL,Category
0,Nat Habit Brahmi Matsyakshi Summer Dasabuti Ba...,Nat,435,370,15%,N/A,218,https://www.nykaa.com/nat-habit-brahmi-matsyak...,Baby
1,Max Care Virgin Coconut Oil Cold Pressed,Max,325,302,7%,N/A,104248,https://www.nykaa.com/max-care-virgin-cold-pre...,Baby
2,Aveeno Baby Daily Moisture Wash & Shampoo - Na...,Aveeno,1379,1299,6%,N/A,5227,https://www.nykaa.com/aveeno-baby-daily-moistu...,Baby
3,Orimii Bump Hydrating Body Butter for Reducing...,Orimii,695,660,5%,N/A,432,https://www.nykaa.com/orimii-bump-hydrating-wh...,Baby
4,Forest Essentials Baby Body Massage Oil Dasapu...,Forest,N/A,N/A,N/A,N/A,169,https://www.nykaa.com/forest-essentials-baby-b...,Baby
5,Bare Anatomy Junior Gentle Cleansing Shampoo,Bare,978,860,12%,N/A,4501,https://www.nykaa.com/bare-anatomy-junior-gent...,Baby
6,Aveeno Baby Daily Moisture Lotion | Oatmeal Fa...,Aveeno,1650,1525,8%,N/A,286,https://www.nykaa.com/aveeno-baby-daily-moistu...,Baby
7,Aveeno Baby Daily Moisture Lotion | Oatmeal Fa...,Aveeno,1050,949,10%,N/A,1806,https://www.nykaa.com/aveeno-baby-daily-moistu...,Baby
8,BABY FOREST Neer 99.9% Water Baby Wipes - Pack...,BABY,898,602,33%,N/A,4,https://www.nykaa.com/baby-forest-neer-99-9per...,Baby
9,Cetaphil Baby Wash & Shampoo With Organic Cale...,Cetaphil,1499,1379,8%,N/A,1536,https://www.nykaa.com/cetaphil-baby-wash-shamp...,Baby
